In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
import torch.optim as optim
from timm.models.layers import DropPath, to_2tuple, trunc_normal_
from collections import OrderedDict
import timm
import os
import kagglehub
from torch.utils.data import DataLoader, IterableDataset
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from tqdm import tqdm
import shutil
import sys
import zipfile
import tarfile
from pathlib import Path
from datasets import load_dataset
from PIL import Image
import io

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [ ]:
class UniformQuantizer(nn.Module):
    def __init__(self, n_bits=8, symmetric=True, channel_wise=False, momentum=0.1, ch_axis=0, eps=1e-8):
        super().__init__()
        self.n_bits = int(n_bits)
        self.symmetric = bool(symmetric)
        self.channel_wise = bool(channel_wise)
        self.momentum = float(momentum)
        self.ch_axis = int(ch_axis)
        self.eps = float(eps)

        if self.symmetric:
            self.qmin = -(2 ** (self.n_bits - 1))
            self.qmax = 2 ** (self.n_bits - 1) - 1
        else:
            self.qmin = 0
            self.qmax = 2 ** self.n_bits - 1

        # safe neutral init
        self.register_buffer("scale", torch.tensor(1.0, dtype=torch.float32))
        self.register_buffer("zero_point", torch.tensor(0.0, dtype=torch.float32))
        #self.register_buffer("scale", torch.tensor(1.0, dtype=torch.float32), persistent=False) #to prevent it from appearing in the keys
        #self.register_buffer("zero_point", torch.tensor(0.0, dtype=torch.float32), persistent=False) #to prevent it from appearing in the keys
        self._calibrated = False

    def _ensure_shape(self, sample_scale):
        # ensure self.scale has the same shape as sample_scale
        if self.scale.shape != sample_scale.shape:
            with torch.no_grad():
                s = torch.zeros_like(sample_scale, device=self.scale.device, dtype=self.scale.dtype)
                z = torch.zeros_like(sample_scale, device=self.zero_point.device, dtype=self.zero_point.dtype)
                # replace buffers in-place (so state_dict keeps names)
                self.scale = s
                self.zero_point = z
            self._calibrated = True

    def calculate_qparams(self, x):
        if self.channel_wise:
            dims = [i for i in range(x.ndim) if i != self.ch_axis]
            x_min = x.amin(dim=dims, keepdim=True)
            x_max = x.amax(dim=dims, keepdim=True)
        else:
            x_min = x.min()
            x_max = x.max()

        if self.symmetric:
            x_absmax = torch.maximum(x_min.abs(), x_max.abs())
            qmax_sym = float(2 ** (self.n_bits - 1) - 1)
            scale = (x_absmax / qmax_sym).clamp(min=self.eps)
            zero_point = torch.zeros_like(scale)
        else:
            denom = float(self.qmax - self.qmin)
            scale = ((x_max - x_min) / denom).clamp(min=self.eps)
            zero_point = (self.qmin - torch.round(x_min / scale)).clamp(min=self.qmin, max=self.qmax)

        return scale, zero_point

    @torch.no_grad()
    def init_from_tensor(self, x):
        """Initialize scale/zero_point from a sample tensor (e.g. weights)."""
        scale, zp = self.calculate_qparams(x)
        self._ensure_shape(scale)
        # copy into buffers in-place
        self.scale.copy_(scale.detach().to(self.scale.device))
        self.zero_point.copy_(zp.detach().to(self.zero_point.device))
        self._calibrated = True

    def forward(self, x, simulate_int=False):
        # training: fake quantize; eval: dequantized floats by default
        if self.training:
            scale_b, zp_b = self.calculate_qparams(x)
            if self.channel_wise:
                self._ensure_shape(scale_b)
            # EMA update (in-place)
            with torch.no_grad():
                s_new = scale_b.detach().to(self.scale.device)
                z_new = zp_b.detach().to(self.zero_point.device)
                # in-place update to preserve buffer objects in state_dict
                if self.scale.shape == s_new.shape:
                    self.scale.mul_(1.0 - self.momentum).add_(self.momentum * s_new)
                    self.zero_point.mul_(1.0 - self.momentum).add_(self.momentum * z_new)
                else:
                    # broadcast-safe assignment
                    self.scale.copy_((1.0 - self.momentum) * self.scale + self.momentum * s_new)
                    self.zero_point.copy_((1.0 - self.momentum) * self.zero_point + self.momentum * z_new)

            # Fake quantize (STE-like)
            x_int = torch.round(x / self.scale + self.zero_point)
            x_int = torch.clamp(x_int, self.qmin, self.qmax)
            x_dequant = self.scale * (x_int - self.zero_point)
            # final clamp to avoid inf/nan (extra safety)
            x_dequant = torch.nan_to_num(x_dequant, nan=0.0, posinf=1e6, neginf=-1e6)
            return x_dequant

        else:
            # eval: default -> dequantized floats (safe)
            x_int = torch.round(x / self.scale + self.zero_point)
            x_int = torch.clamp(x_int, self.qmin, self.qmax)
            x_dequant = self.scale * (x_int - self.zero_point)
            return x_dequant if not simulate_int else x_int.to(torch.int8)

class QuantizedLinear(nn.Module):
    """
    Drop-in replacement for nn.Linear that uses UniformQuantizer:
     - per-tensor activation quant (act_quant)
     - per-channel weight quant (wt_quant, ch_axis=0)
    Notes:
      - training: fake-quantized floats are used (common QAT)
      - eval(simulate_int=False): dequantized floats are used (safe validation)
      - eval(simulate_int=True): integers (int8) are returned by quantizers; user must run hardware int8 matmul
    """
    def __init__(self, in_features, out_features, bias=True, n_bits=8, symmetric=True):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        # activation quant: per-tensor (axis doesn't matter)
        self.act_quant = UniformQuantizer(n_bits=n_bits, symmetric=symmetric, channel_wise=False)
        # weight quant: per-channel on output channels (axis 0 in [out, in])
        self.wt_quant = UniformQuantizer(n_bits=n_bits, symmetric=symmetric, channel_wise=True, ch_axis=0)

    def init_weight_quant(self):
        """Call once after the layer is created/loaded with pretrained weights to init weight scales."""
        self.wt_quant.init_from_tensor(self.linear.weight.data)

    def forward(self, x, simulate_int=False):
        # simulate_int only matters in eval mode; in training we ignore simulate_int
        x_q = self.act_quant(x, simulate_int=simulate_int if not self.training else False)
        # quantize weights - in training returns dequantized floats (fake quant)
        w_q = self.wt_quant(self.linear.weight, simulate_int=simulate_int if not self.training else False)

        # If simulate_int=True and eval: x_q and w_q may be int8; here we assume we don't have int8 matmul,
        # so convert to dequantized float form. But UniformQuantizer returns ints only if simulate_int=True.
        if not self.training and simulate_int:
            # In this branch you would call an int8 GEMM on hardware.
            # As fallback, dequantize to float (so code doesn't break):
            # (but prefer to run true int8 GEMM on deployment).
            x_q = x_q.float() if x_q.dtype.is_floating_point else (self.act_quant.scale * (x_q.float() - self.act_quant.zero_point))
            w_q = w_q.float() if w_q.dtype.is_floating_point else (self.wt_quant.scale * (w_q.float() - self.wt_quant.zero_point))

        # final matmul (using float tensor, either fake-quantized or dequantized)
        out = F.linear(x_q, w_q, self.linear.bias)
        return out

In [ ]:
# we replace all nn.linear by QuantizedLinear that we implement
class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, norm_layer=nn.BatchNorm1d, out_features=None, act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        # self.fc1 = nn.Linear(in_features, hidden_features)
        self.fc1 = QuantizedLinear(in_features, hidden_features)

        self.norm1 = norm_layer(hidden_features)
        self.act = act_layer()
        # self.fc2 = nn.Linear(hidden_features, out_features)
        self.fc2 = QuantizedLinear(hidden_features, out_features)
        self.norm2 = norm_layer(out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = x.transpose(1, 2)
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = x.transpose(1, 2)
        x = self.norm2(x)
        x = x.transpose(1, 2)
        x = self.drop(x)
        return x


def window_partition(x, window_size):
    """
    Args:
        x: (B, H, W, C)
        window_size (int): window size

    Returns:
        windows: (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    Args:
        windows: (num_windows*B, window_size, window_size, C)
        window_size (int): Window size
        H (int): Height of image
        W (int): Width of image

    Returns:
        x: (B, H, W, C)
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x


class WindowAttention(nn.Module):
    """ Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.

    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): The height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set
        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0
        proj_drop (float, optional): Dropout ratio of output. Default: 0.0
    """

    def __init__(self, dim, window_size, num_heads, qkv_bias=True, qk_scale=None, attn_drop=0., proj_drop=0.):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))  # 2*Wh-1 * 2*Ww-1, nH

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index, persistent=False)

        # self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.qkv = QuantizedLinear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        # self.proj = nn.Linear(dim, dim)
        self.proj = QuantizedLinear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        """
        Args:
            x: input features with shape of (num_windows*B, N, C)
            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None
            simulate_int: bool, whether to simulate integer inference (only matters in eval mode)
        """
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            attn = self.softmax(attn)
        else:
            attn = self.softmax(attn)

        # Apply Log2 quantization to attention weights (only in eval mode with simulate_int=True)
        # In training, log_quantizer acts as identity. In eval with simulate_int=False, it dequantizes to float.
        # We need to ensure the output of log_quantizer is float for the matmul if simulating int.
        # attn = self.log_quantizer(attn)

        # # Ensure attn and v are floats for the matrix multiplication when simulating int inference
        # if not self.training and simulate_int:
        #      attn = attn.float() if attn.dtype.is_floating_point else (self.log_quantizer.scale * (attn.float() - self.log_quantizer.zero_point)) # Log2 quantizer output is quantized, but the math here expects float
        #      v = v.float() # Value tensor from QuantizedLinear might be int8, convert to float

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class SwinTransformerBlock(nn.Module):
    """ Swin Transformer Block.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resulotion.
        num_heads (int): Number of attention heads.
        window_size (int): Window size.
        shift_size (int): Shift size for SW-MSA.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, dim, input_resolution, num_heads, window_size=7, shift_size=0,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0., drop_path=0.,
                 act_layer=nn.GELU, norm_layer=nn.BatchNorm1d,
                 fused_window_process=False):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim, window_size=to_2tuple(self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, qk_scale=qk_scale, attn_drop=attn_drop, proj_drop=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)

        if self.shift_size > 0:
            # calculate attention mask for SW-MSA
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
            h_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1

            mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask, persistent=False)
        self.fused_window_process = fused_window_process

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x
        x = x.transpose(1, 2)
        x = self.norm1(x)
        x = x.transpose(1, 2)
        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
                # partition windows
                x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
            else:
                x_windows = WindowProcess.apply(x, B, H, W, C, -self.shift_size, self.window_size)
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        # reverse cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
                x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
            else:
                x = WindowProcessReverse.apply(attn_windows, B, H, W, C, self.shift_size, self.window_size)
        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x
        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)

        # FFN
        x = x.transpose(1, 2)
        x = self.norm2(x)
        x = x.transpose(1, 2)
        x = x + self.drop_path(self.mlp(x))

        return x


class PatchMerging(nn.Module):
    """ Patch Merging Layer.

    Args:
        input_resolution (tuple[int]): Resolution of input feature.
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(self, input_resolution, dim, norm_layer=nn.BatchNorm1d):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        # self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.reduction = QuantizedLinear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x):
        """
        x: B, H*W, C
        simulate_int: bool, whether to simulate integer inference (only matters in eval mode)
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."

        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = x.transpose(1, 2)
        x = self.norm(x)
        x = x.transpose(1, 2)
        # Pass simulate_int to the reduction module
        x = self.reduction(x)

        return x


class BasicLayer(nn.Module):
    """ A basic Swin Transformer layer for one stage.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution.
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, dim, input_resolution, depth, num_heads, window_size,
                 mlp_ratio=4., qkv_bias=True, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., norm_layer=nn.BatchNorm1d, downsample=None, use_checkpoint=False,
                 fused_window_process=False):

        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 drop=drop, attn_drop=attn_drop,
                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                                 norm_layer=norm_layer,
                                 fused_window_process=fused_window_process)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim, norm_layer=norm_layer)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            if self.use_checkpoint:
                # Pass simulate_int to the block
                x = checkpoint.checkpoint(blk, x)
            else:
                 # Pass simulate_int to the block
                x = blk(x)
        if self.downsample is not None:
            # Pass simulate_int to the downsample layer
            x = self.downsample(x)
        return x


class PatchEmbed(nn.Module):
    """ Image to Patch Embedding

    Args:
        img_size (int): Image size.  Default: 224.
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.num_patches = patches_resolution[0] * patches_resolution[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        if norm_layer is not None:
            self.norm = norm_layer(embed_dim)
        else:
            self.norm = None

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        assert H == self.img_size[0] and W == self.img_size[1], \
            f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)  # B Ph*Pw C
        if self.norm is not None:
            x = x.transpose(1, 2)                     # (B, C, L)
            x = self.norm(x)
            x = x.transpose(1, 2)
        return x


class SwinTransformer(nn.Module):
    """ Swin Transformer
        A PyTorch impl of : `Swin Transformer: Hierarchical Vision Transformer using Shifted Windows`  -
          https://arxiv.org/pdf/2103.14030

    Args:
        img_size (int | tuple(int)): Input image size. Default 224
        patch_size (int | tuple(int)): Patch size. Default: 4
        in_chans (int): Number of input image channels. Default: 3
        num_classes (int): Number of classes for classification head. Default: 1000
        embed_dim (int): Patch embedding dimension. Default: 96
        depths (tuple(int)): Depth of each Swin Transformer layer.
        num_heads (tuple(int)): Number of attention heads in different layers.
        window_size (int): Window size. Default: 7
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4
        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float): Override default qk scale of head_dim ** -0.5 if set. Default: None
        drop_rate (float): Dropout rate. Default: 0
        attn_drop_rate (float): Attention dropout rate. Default: 0
        drop_path_rate (float): Stochastic depth rate. Default: 0.1
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False
        patch_norm (bool): If True, add normalization after patch embedding. Default: True
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self, img_size=224, patch_size=4, in_chans=3, num_classes=1000,
                 embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24],
                 window_size=7, mlp_ratio=4., qkv_bias=True, qk_scale=None,
                 drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
                 norm_layer=nn.BatchNorm1d, ape=False, patch_norm=True,
                 use_checkpoint=False, fused_window_process=False, **kwargs):
        super().__init__()

        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=in_chans, embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None)
        num_patches = self.patch_embed.num_patches
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # absolute position embedding
        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias, qk_scale=qk_scale,
                               drop=drop_rate, attn_drop=attn_drop_rate,
                               drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                               norm_layer=norm_layer,
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                               use_checkpoint=use_checkpoint,
                               fused_window_process=fused_window_process)
            self.layers.append(layer)

        self.norm = norm_layer(self.num_features)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        # self.head = nn.Sequential(OrderedDict([("fc", nn.Linear(self.num_features, num_classes))])) if num_classes > 0 else nn.Identity()
        self.head = nn.Sequential(OrderedDict([("fc", QuantizedLinear(self.num_features, num_classes))])) if num_classes > 0 else nn.Identity()

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, QuantizedLinear):
            m.init_weight_quant()

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'absolute_pos_embed'}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'relative_position_bias_table'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        for layer in self.layers:
            # Pass simulate_int to the basic layer
            x = layer(x)

        x = x.transpose(1, 2)
        x = self.norm(x)  # B L C
        x = x.transpose(1, 2)
        x = self.avgpool(x.transpose(1, 2))  # B C 1
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        # Pass simulate_int to the head
        x = self.head(x)
        return x

In [ ]:
custom_model = SwinTransformer(
    img_size=224,
    patch_size=4,
    in_chans=3,
    num_classes=1000,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[3, 6, 12, 24],
    ape=False,
    patch_norm=True,
    use_checkpoint=False,
    fused_window_process=False,
)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [ ]:
#to add "linear" to keys
import re

def convert_timm_to_custom_keys(orig_state):
    """
    Convert timm-style keys (e.g. 'layers.0.blocks.0.mlp.fc1.weight')
    into the custom format used in your checkpoint (e.g. '...mlp.fc1.linear.weight').

    Rules implemented (based on your lists):
      - .mlp.fc1.  -> .mlp.fc1.linear.
      - .mlp.fc2.  -> .mlp.fc2.linear.
      - .attn.proj. -> .attn.proj.linear.
      - .attn.qkv. -> .attn.qkv.linear.
      - head.fc.   -> head.fc.linear.
      - .downsample.reduction. -> .downsample.reduction.linear.
    The function avoids re-applying `.linear.` if it's already present.
    """
    new_state = {}
    collisions = []

    for k, v in orig_state.items():
        new_k = k

        # If key already contains '.linear.' we assume it's already in the target format
        if '.linear.' not in new_k:
            # do targeted replacements only
            # mlp fc layers
            if '.mlp.fc1.' in new_k:
                new_k = new_k.replace('.mlp.fc1.', '.mlp.fc1.linear.')
            if '.mlp.fc2.' in new_k:
                new_k = new_k.replace('.mlp.fc2.', '.mlp.fc2.linear.')

            # attention projection / qkv
            if '.attn.proj.' in new_k:
                new_k = new_k.replace('.attn.proj.', '.attn.proj.linear.')
            if '.attn.qkv.' in new_k:
                new_k = new_k.replace('.attn.qkv.', '.attn.qkv.linear.')

            # head fc
            if 'head.fc.' in new_k:
                # 'head.fc.' appears at start of key component: replace only that occurrence
                new_k = new_k.replace('head.fc.', 'head.fc.linear.')

            # downsample reduction (these were changed to reduction.linear in your custom keys)
            if '.downsample.reduction.' in new_k:
                new_k = new_k.replace('.downsample.reduction.', '.downsample.reduction.linear.')

            # (OPTIONAL) if you later find other modules that require ".linear" insertion
            # add them here the same way, e.g.:
            # if '.some_module.xyz.' in new_k:
            #     new_k = new_k.replace('.some_module.xyz.', '.some_module.xyz.linear.')

        # Check for collisions (two different orig keys mapped to same new key)
        if new_k in new_state and new_state[new_k] is not v:
            collisions.append((new_k, k))

        new_state[new_k] = v

    if collisions:
        # This prints to stdout; in a script you may want to raise or log instead.
        print("Warning: key collisions detected (multiple original keys mapped to same new key):")
        for coll in collisions:
            print("  mapped key:", coll[0], "from original key:", coll[1])

    return new_state

In [ ]:
#to see all keys and compare it
orig_state = torch.load("/content/drive/MyDrive/custom_model_weights6.pth")
new_state = convert_timm_to_custom_keys(orig_state)

ref_ds = sorted(k for k in new_state)
cust_ds = sorted(k for k in custom_model.state_dict().keys())

print("timm keys:", ref_ds)
print("custom keys:", cust_ds)

timm keys: ['head.fc.linear.bias', 'head.fc.linear.weight', 'layers.0.blocks.0.attn.proj.linear.bias', 'layers.0.blocks.0.attn.proj.linear.weight', 'layers.0.blocks.0.attn.qkv.linear.bias', 'layers.0.blocks.0.attn.qkv.linear.weight', 'layers.0.blocks.0.attn.relative_position_bias_table', 'layers.0.blocks.0.mlp.fc1.linear.bias', 'layers.0.blocks.0.mlp.fc1.linear.weight', 'layers.0.blocks.0.mlp.fc2.linear.bias', 'layers.0.blocks.0.mlp.fc2.linear.weight', 'layers.0.blocks.0.mlp.norm1.bias', 'layers.0.blocks.0.mlp.norm1.num_batches_tracked', 'layers.0.blocks.0.mlp.norm1.running_mean', 'layers.0.blocks.0.mlp.norm1.running_var', 'layers.0.blocks.0.mlp.norm1.weight', 'layers.0.blocks.0.mlp.norm2.bias', 'layers.0.blocks.0.mlp.norm2.num_batches_tracked', 'layers.0.blocks.0.mlp.norm2.running_mean', 'layers.0.blocks.0.mlp.norm2.running_var', 'layers.0.blocks.0.mlp.norm2.weight', 'layers.0.blocks.0.norm1.bias', 'layers.0.blocks.0.norm1.num_batches_tracked', 'layers.0.blocks.0.norm1.running_mean', 

In [ ]:
missing, unexpected = custom_model.load_state_dict(new_state, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: []
Unexpected keys: []


In [ ]:
orig_state = torch.load("/content/drive/MyDrive/weights_after_QuantizedLinear.pth")

missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

Missing keys: ['layers.0.blocks.0.attn.qkv.act_quant.scale', 'layers.0.blocks.0.attn.qkv.act_quant.zero_point', 'layers.0.blocks.0.attn.qkv.wt_quant.scale', 'layers.0.blocks.0.attn.qkv.wt_quant.zero_point', 'layers.0.blocks.0.attn.proj.act_quant.scale', 'layers.0.blocks.0.attn.proj.act_quant.zero_point', 'layers.0.blocks.0.attn.proj.wt_quant.scale', 'layers.0.blocks.0.attn.proj.wt_quant.zero_point', 'layers.0.blocks.0.mlp.fc1.act_quant.scale', 'layers.0.blocks.0.mlp.fc1.act_quant.zero_point', 'layers.0.blocks.0.mlp.fc1.wt_quant.scale', 'layers.0.blocks.0.mlp.fc1.wt_quant.zero_point', 'layers.0.blocks.0.mlp.fc2.act_quant.scale', 'layers.0.blocks.0.mlp.fc2.act_quant.zero_point', 'layers.0.blocks.0.mlp.fc2.wt_quant.scale', 'layers.0.blocks.0.mlp.fc2.wt_quant.zero_point', 'layers.0.blocks.1.attn.qkv.act_quant.scale', 'layers.0.blocks.1.attn.qkv.act_quant.zero_point', 'layers.0.blocks.1.attn.qkv.wt_quant.scale', 'layers.0.blocks.1.attn.qkv.wt_quant.zero_point', 'layers.0.blocks.1.attn.proj.

In [ ]:
#go to link will appear to you after running this cell and create a token
#وانته بتعملها اعمل صح علي اي حاجه تقدر تعملها صح
#paste token code here
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `swin` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate whe

In [ ]:
# ----------------------------------------------------------------------------
# 1) Load ImageNet-1k training set (streaming mode)
# ----------------------------------------------------------------------------
hf_dataset = load_dataset("ILSVRC/imagenet-1k", split="train", streaming=True)


train_stream = hf_dataset.take(100_000)
val_stream   = hf_dataset.skip(100_000).take(15_000)
test_stream = hf_dataset.skip(115_000).take(10_000)


transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def collate_fn(batch):
    """Convert HF samples into tensors"""
    images, labels = [], []
    for sample in batch:
        img = sample["image"]
        if not isinstance(img, Image.Image):  # if numpy array
            img = Image.fromarray(img)
        img = img.convert("RGB")  # <-- ensure 3 channels
        images.append(transform(img))
        labels.append(sample["label"])
    return torch.stack(images), torch.tensor(labels)


train_loader = DataLoader(train_stream, batch_size=64, collate_fn=collate_fn)
val_loader   = DataLoader(val_stream,   batch_size=64, collate_fn=collate_fn)
test_loader = DataLoader(test_stream, batch_size=64, collate_fn=collate_fn)


def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        torch.autograd.set_detect_anomaly(True)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (i + 1) % 50 == 0:
            print(f"[Epoch {epoch} | Step {i+1}] "
                  f"Loss: {running_loss/(i+1):.4f} | "
                  f"Acc: {100.*correct/total:.2f}%")

    return running_loss / (i+1), 100.*correct/total

def evaluate(model, dataloader, criterion, device):
    model.eval()
    top1, top5, total, total_loss = 0, 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(dataloader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            # Top-1
            _, pred1 = outputs.topk(1, dim=1)
            top1 += (pred1.squeeze() == labels).sum().item()

            # Top-5
            _, pred5 = outputs.topk(5, dim=1)
            top5 += sum([labels[j].item() in pred5[j] for j in range(labels.size(0))])

            total += labels.size(0)

            if (i+1) % 50 == 0:
                print(f"[Val Step {i+1}] "
                      f"Loss: {total_loss/(i+1):.4f} | "
                      f"Top-1: {100.*top1/total:.2f}% | "
                      f"Top-5: {100.*top5/total:.2f}%")

    return 100.*top1/total, 100.*top5/total, total_loss/(i+1)

criterion = nn.CrossEntropyLoss()

device = "cuda" if torch.cuda.is_available() else "cpu"
custom_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/87.6k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

SwinTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (norm): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (layers): ModuleList(
    (0): BasicLayer(
      (blocks): ModuleList(
        (0): SwinTransformerBlock(
          (norm1): BatchNorm1d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (attn): WindowAttention(
            (qkv): QuantizedLinear(
              (linear): Linear(in_features=96, out_features=288, bias=True)
              (act_quant): UniformQuantizer()
              (wt_quant): UniformQuantizer()
            )
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): QuantizedLinear(
              (linear): Linear(in_features=96, out_features=96, bias=True)
              (act_quant): UniformQuantizer()
              (wt_quant): UniformQuantizer()
            )
            (proj_drop

In [ ]:
optimizer = optim.Adam(custom_model.parameters(), lr=1e-3)

EPOCHS = 5  # just for demo, set higher for real training
for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = train_one_epoch(custom_model, train_loader, criterion, optimizer, device, epoch)
    print(f"\n[Epoch {epoch}] Training Loss: {train_loss:.4f} | Training Acc: {train_acc:.2f}%")

    val_top1, val_top5, val_loss = evaluate(custom_model, val_loader, criterion, device)
    print(f"[Epoch {epoch}] Validation → Loss: {val_loss:.4f} | Top-1: {val_top1:.2f}% | Top-5: {val_top5:.2f}%\n")

[Epoch 1 | Step 50] Loss: 0.5311 | Acc: 86.00%


KeyboardInterrupt: 

In [ ]:
#testing
top1, top5, avg_loss = evaluate(custom_model, test_loader, criterion, device)
print(f"\nValidation Accuracy → Top-1: {top1:.2f}% | Top-5: {top5:.2f}% | Loss: {avg_loss:.4f}")

[Val Step 50] Loss: 0.6510 | Top-1: 82.19% | Top-5: 96.69%
[Val Step 100] Loss: 0.6638 | Top-1: 82.25% | Top-5: 96.28%
[Val Step 150] Loss: 0.6619 | Top-1: 82.21% | Top-5: 96.27%
[Val Step 200] Loss: 0.6717 | Top-1: 82.56% | Top-5: 95.91%
[Val Step 250] Loss: 0.6697 | Top-1: 82.56% | Top-5: 95.92%
[Val Step 300] Loss: 0.6520 | Top-1: 82.78% | Top-5: 96.22%

Validation Accuracy → Top-1: 82.77% | Top-5: 96.31% | Loss: 0.6492


In [ ]:
#save your work to your drive
torch.save(custom_model.state_dict(), "custom_model_weights7.pth")

drive_path = "/content/drive/MyDrive/custom_model_weights7.pth"
shutil.copy("custom_model_weights7.pth", drive_path)

In [ ]:
# Load your saved work
orig_state = torch.load("/content/drive/MyDrive/add_name.pth")
missing, unexpected = custom_model.load_state_dict(orig_state, strict=False)

print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

In [ ]:
#freezing
for name, param in custom_model.named_parameters():
    if "norm" in name.lower() or name.startswith("head.fc") or name.startswith("layers.3"):
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = [n for n,p in custom_model.named_parameters() if p.requires_grad]
frozen    = [n for n,p in custom_model.named_parameters() if not p.requires_grad]
print(f"✨ {len(trainable)} params left trainable (norm layers):")
print(trainable)
print(f"🥶 {len(frozen)} params frozen:")
print(frozen)  # show a few

In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
#testing
top1, top5, avg_loss = evaluate(custom_model, test_loader, criterion, device)
print(f"\nValidation Accuracy → Top-1: {top1:.2f}% | Top-5: {top5:.2f}% | Loss: {avg_loss:.4f}")

[Val Step 50] Loss: 0.6554 | Top-1: 82.84% | Top-5: 96.31%
[Val Step 100] Loss: 0.6639 | Top-1: 82.75% | Top-5: 96.03%
[Val Step 150] Loss: 0.6463 | Top-1: 82.94% | Top-5: 96.40%

Validation Accuracy → Top-1: 82.92% | Top-5: 96.49% | Loss: 0.6427


In [ ]:
optimizer = optim.Adam(custom_model.parameters(), lr=1e-7)

EPOCHS = 1  # just for demo, set higher for real training
for epoch in range(1, EPOCHS+1):
    train_loss, train_acc = train_one_epoch(custom_model, train_loader, criterion, optimizer, device, epoch)
    print(f"\n[Epoch {epoch}] Training Loss: {train_loss:.4f} | Training Acc: {train_acc:.2f}%")

    val_top1, val_top5, val_loss = evaluate(custom_model, val_loader, criterion, device)
    print(f"[Epoch {epoch}] Validation → Loss: {val_loss:.4f} | Top-1: {val_top1:.2f}% | Top-5: {val_top5:.2f}%\n")

[Epoch 1 | Step 50] Loss: 0.6744 | Acc: 82.34%
[Epoch 1 | Step 100] Loss: 0.6291 | Acc: 83.27%
[Epoch 1 | Step 150] Loss: 0.5953 | Acc: 84.04%
[Epoch 1 | Step 200] Loss: 0.5843 | Acc: 84.26%
[Epoch 1 | Step 250] Loss: 0.5776 | Acc: 84.33%
[Epoch 1 | Step 300] Loss: 0.5748 | Acc: 84.49%
[Epoch 1 | Step 350] Loss: 0.5672 | Acc: 84.70%
[Epoch 1 | Step 400] Loss: 0.5664 | Acc: 84.83%
[Epoch 1 | Step 450] Loss: 0.5625 | Acc: 85.02%
[Epoch 1 | Step 500] Loss: 0.5607 | Acc: 85.11%
[Epoch 1 | Step 550] Loss: 0.5601 | Acc: 85.14%
[Epoch 1 | Step 600] Loss: 0.5578 | Acc: 85.20%
[Epoch 1 | Step 650] Loss: 0.5551 | Acc: 85.25%
[Epoch 1 | Step 700] Loss: 0.5536 | Acc: 85.27%
[Epoch 1 | Step 750] Loss: 0.5510 | Acc: 85.31%
[Epoch 1 | Step 800] Loss: 0.5500 | Acc: 85.33%
[Epoch 1 | Step 850] Loss: 0.5469 | Acc: 85.41%
[Epoch 1 | Step 900] Loss: 0.5453 | Acc: 85.45%
[Epoch 1 | Step 950] Loss: 0.5424 | Acc: 85.50%
[Epoch 1 | Step 1000] Loss: 0.5388 | Acc: 85.60%
[Epoch 1 | Step 1050] Loss: 0.5370 | Acc

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


[Epoch 1 | Step 1500] Loss: 0.5110 | Acc: 86.40%
[Epoch 1 | Step 1550] Loss: 0.5059 | Acc: 86.54%

[Epoch 1] Training Loss: 0.5062 | Training Acc: 86.55%
[Val Step 50] Loss: 0.6570 | Top-1: 82.50% | Top-5: 96.28%
[Val Step 100] Loss: 0.6415 | Top-1: 82.80% | Top-5: 96.42%
[Val Step 150] Loss: 0.6459 | Top-1: 82.64% | Top-5: 96.61%
[Val Step 200] Loss: 0.6463 | Top-1: 82.64% | Top-5: 96.66%
[Epoch 1] Validation → Loss: 0.6464 | Top-1: 82.62% | Top-5: 96.62%



In [ ]:
#save your work to your drive
torch.save(custom_model.state_dict(), "custom_model_weights14.pth")

drive_path = "/content/drive/MyDrive/custom_model_weights14.pth"
shutil.copy("custom_model_weights14.pth", drive_path)

'/content/drive/MyDrive/custom_model_weights14.pth'